# Example: EWLS Engine Replay

In this example, we replay the Cobb-Douglas rebalancing engine with **exponentially weighted least squares (EWLS)** parameter updates. Instead of using frozen SIM parameters from a historical calibration window, the engine re-estimates $(\alpha_i, \beta_i, \sigma_{\varepsilon,i})$ daily as new market data arrives. We compare the frozen and online engines on the Session 2 Monte Carlo ensemble, then zoom into single paths to see what one realized future looks like.

> __Learning Objectives:__
>
> * __Run the EWLS engine on a Monte Carlo ensemble:__ Replay the engine with online parameter updates across hundreds of synthetic paths and compare the distributional performance against the frozen-parameter baseline. Read the median terminal wealth, drawdown, and Sharpe statistics to judge whether online adaptation tightens the outcome distribution.
> * __Visualize parameter drift on single paths:__ Select individual paths and plot how the engine's SIM intercept and slope estimates evolve over the trading horizon. Use these trajectories to see the engine adapting to regime shifts that the frozen baseline cannot react to.
> * __Sweep the half-life for bias-variance trade-off:__ Run the EWLS engine at three half-life values and compare how quickly the estimates track regime shifts versus how much noise they introduce. Identify the half-life that balances responsiveness against estimation noise on a single realized path.

Let's see what happens when the engine's world-model updates itself.

___

## Setup, Data and Prerequisites

In [ ]:
include("Include.jl");

### Constants


In [ ]:
# EWLS replay configuration
B₀ = 10_000.0
Δt = 1.0 / 252.0
L_short = 21
L_long = 63
L_growth = 10
GAIN = 10.0
offset = L_short + L_long
SCENARIO_SEED = 2026
N_ENSEMBLE_PATHS = 500
N_ACTIVE_TRADING_DAYS = 252
EWLS_HALF_LIFE = 63.0
EWLS_PRIOR_WEIGHT = 63.0
SAMPLE_PATH_SEED = 42
TRIGGER_MAX_DRAWDOWN = 0.15
TRIGGER_MAX_TURNOVER = 0.50


Load the Session 1 artifacts (SIM parameter estimates, ticker universe) and the pre-trained surrogates needed for scenario generation. The artifacts include [`MySIMParameterEstimate`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session1/#eCornellAIFinance.MySIMParameterEstimate) records and the [`MyMarketSurrogateModel`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session1/#eCornellAIFinance.MyMarketSurrogateModel) and [`MyPortfolioSurrogateModel`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session1/#eCornellAIFinance.MyPortfolioSurrogateModel) calibrated in Session 1. In the code block below, we return: `my_tickers::Vector{String}`, `sim_estimates::Vector{MySIMParameterEstimate}`, `sim_params::Dict{String,Tuple{Float64,Float64,Float64}}`, `σ_m::Float64`, `market_model::MyMarketSurrogateModel`, `portfolio::MyPortfolioSurrogateModel`, `calib::Dict{String,Any}`, `start_prices::Dict{String,Float64}`, and `N::Int`, all needed by [the `generate_hybrid_scenario(...)` function](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.generate_hybrid_scenario).

In [ ]:
my_tickers, sim_estimates, sim_params, σ_m, market_model, portfolio, calib, start_prices, N = let
    # --- Step 1: Load S1 artifacts ---
    minvar = load_results(joinpath(_PATH_TO_DATA_S1, "minvar-allocation.jld2"));
    my_tickers    = minvar["my_tickers"]::Vector{String};
    sim_estimates = minvar["sim_estimates"];
    g_f           = haskey(minvar, "g_f") ? Float64(minvar["g_f"]) : Float64(minvar["r_f"]);

    # --- Step 2: Build SIM parameter dictionary ---
    sim_params = Dict{String,Tuple{Float64,Float64,Float64}}(
        e.ticker => (e.α, e.β, e.σ_ε) for e in sim_estimates
    );
    σ_m = minvar["sigma_market"]::Float64;

    # --- Step 3: Load surrogates and calibration ---
    market_model = MyMarketSurrogateModel();
    portfolio    = MyPortfolioSurrogateModel();
    calib        = load_results(joinpath(_PATH_TO_DATA_S1, "sim-parameter-estimates.jld2"));
    snap        = MyCurrentPrices();
    snap_lookup = Dict(snap["tickers"] .=> snap["prices"]);
    start_prices = Dict(t => snap_lookup[t] for t in my_tickers);

    # --- Step 4: Constants ---
    N         = length(my_tickers);

    println("Loaded $(N) tickers: $(my_tickers)")
    println("Risk-free growth rate: $(round(g_f*100, digits=2))%/yr")
    my_tickers, sim_estimates, sim_params, σ_m, market_model, portfolio, calib, start_prices, N
end

___
## Task 1: Frozen vs. Online on the Monte Carlo Ensemble
In this task, we generate a 500-path hybrid scenario using the same infrastructure as Session 2, then run two engines across all paths: one with frozen SIM parameters and one with EWLS online updates (half-life = 63 days). The comparison reveals whether online adaptation improves the distribution of outcomes.

> __What should we see?__
>
> The EWLS engine should show a tighter wealth distribution (lower variance) and potentially higher median Sharpe, because its preference weights adapt to regime shifts that the frozen engine misses. The improvement is largest on paths that include regime transitions.

The code block below returns `scenario::MyBacktestScenario` (500-path × $T$-step hybrid futures, reused by downstream examples via the locked seed) and `frozen_bt::MyBacktestResult` (the frozen-SIM engine baseline that the EWLS run is compared against). Internally, the cell calls [`backtest_engine(...)`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.backtest_engine) for the frozen baseline and [`replay_engine_ewls(...)`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.replay_engine_ewls) per path for the online engine; the result types are [`MyBacktestScenario`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.MyBacktestScenario) and [`MyBacktestResult`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.MyBacktestResult).

In [ ]:
scenario, frozen_bt = let
    # --- Step 1: Generate the 500-path scenario ---
    n_paths = N_ENSEMBLE_PATHS;
    n_trading_days = N_ACTIVE_TRADING_DAYS;
    T_total = offset + n_trading_days;

    Random.seed!(SCENARIO_SEED);
    scenario = generate_hybrid_scenario(
        market_model, portfolio, calib, my_tickers;
        n_paths = n_paths, n_steps = T_total,
        start_prices = start_prices, label = "S3 EWLS Ensemble", seed = SCENARIO_SEED);

    # --- Step 2: Run the frozen-parameter engine via backtest_engine ---
    rules_params = (max_drawdown = TRIGGER_MAX_DRAWDOWN, max_turnover = TRIGGER_MAX_TURNOVER);
    frozen_bt = backtest_engine(scenario, my_tickers, sim_params, rules_params;
        B₀ = B₀, offset = offset);

    # --- Step 3: Run EWLS engine across all paths ---
    n_trading = T_total - offset;
    ewls_final_wealth = zeros(n_paths);
    ewls_max_dd = zeros(n_paths);
    ewls_sharpe = zeros(n_paths);

    for p in 1:n_paths
        mkt = scenario.market_paths[p, :];
        pmatrix = zeros(T_total, N + 1);
        pmatrix[:, 1] = 1:T_total;
        for k in 1:N
            pmatrix[:, k + 1] = scenario.price_paths[p, :, k];
        end

        result = replay_engine_ewls(pmatrix, mkt, my_tickers, sim_params, rules_params;
            B₀ = B₀, offset = offset, half_life = EWLS_HALF_LIFE, prior_weight = EWLS_PRIOR_WEIGHT,
            N_short = L_short, N_long = L_long, GAIN = GAIN, N_growth = L_growth);

        w = result.wealth;
        ewls_final_wealth[p] = w[end];
        returns_p = diff(w) ./ w[1:end-1];
        peak_p = accumulate(max, w);
        ewls_max_dd[p] = maximum((peak_p .- w) ./ peak_p);
        vol_p = std(returns_p) * sqrt(252);
        ewls_sharpe[p] = vol_p > 0 ? (w[end]/w[1] - 1.0) / vol_p : 0.0;
    end

    # --- Step 4: Compare distributions ---
    println("="^60)
    println("  FROZEN vs EWLS ENGINE: Distributional Comparison ($(n_paths) paths)")
    println("="^60)
    println("  Metric               Frozen          EWLS (h=63)")
    println("  " * "-"^50)
    println("  Median W/W₀          $(round(median(frozen_bt.final_wealth)/B₀, digits=3))           $(round(median(ewls_final_wealth)/B₀, digits=3))")
    println("  Median Max DD        $(round(median(frozen_bt.max_drawdowns)*100, digits=1))%           $(round(median(ewls_max_dd)*100, digits=1))%")
    println("  Median Sharpe        $(round(median(frozen_bt.sharpe_ratios), digits=3))           $(round(median(ewls_sharpe), digits=3))")
    println("  Fail rate P[W<B₀]   $(round(mean(frozen_bt.final_wealth .< B₀)*100, digits=1))%           $(round(mean(ewls_final_wealth .< B₀)*100, digits=1))%")

    # --- Step 5: Save for downstream examples ---
    save_results(joinpath(_PATH_TO_DATA, "ewls-replay-results.jld2"), Dict(
        "frozen_final_wealth" => frozen_bt.final_wealth,
        "frozen_max_dd" => frozen_bt.max_drawdowns,
        "frozen_sharpe" => frozen_bt.sharpe_ratios,
        "ewls_final_wealth" => ewls_final_wealth,
        "ewls_max_dd" => ewls_max_dd,
        "ewls_sharpe" => ewls_sharpe,
        "n_paths" => n_paths,
        "my_tickers" => my_tickers,
        "sim_params" => sim_params,
    ));
    println("\nSaved results to ewls-replay-results.jld2")
    scenario, frozen_bt
end

The ensemble comparison shows the distributional shift. The next step asks: what does this look like on a single path? We only ever experience one future.

___
## Task 2: Single-Path Replay: One Realized Future
In this task, we select three random paths from the ensemble and replay the EWLS engine on each, plotting the wealth trajectory alongside the SIM parameter drift. This shows how the engine's internal model evolves in response to regime shifts on a specific realization. The replay calls [`replay_engine_ewls(...)`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.replay_engine_ewls) per path.

> __What should we see?__
>
> The $\alpha$ and $\beta$ estimates should drift noticeably when the path crosses a regime boundary (e.g., from bull to bear). The wealth trajectory of the EWLS engine should diverge from the frozen engine precisely at these regime transitions.

In the code block below, we loop over the three sample paths and render a two-panel figure per path: a wealth trajectory above and an α-drift series for the first two tickers below.

In [ ]:
let
    # --- Step 1: Select 3 random paths ---
    Random.seed!(SAMPLE_PATH_SEED);
    sample_paths = sort(rand(1:scenario.n_paths, 3));
    rules_params = (max_drawdown = TRIGGER_MAX_DRAWDOWN, max_turnover = TRIGGER_MAX_TURNOVER);
    T_total = scenario.n_steps;

    for (j, p) in enumerate(sample_paths)
        mkt = scenario.market_paths[p, :];
        pmatrix = zeros(T_total, N + 1);
        pmatrix[:, 1] = 1:T_total;
        for k in 1:N
            pmatrix[:, k + 1] = scenario.price_paths[p, :, k];
        end

        # --- Step 2: Run EWLS engine ---
        result = replay_engine_ewls(pmatrix, mkt, my_tickers, sim_params, rules_params;
            B₀ = B₀, offset = offset, half_life = EWLS_HALF_LIFE, prior_weight = EWLS_PRIOR_WEIGHT,
            N_short = L_short, N_long = L_long, GAIN = GAIN, N_growth = L_growth);

        # --- Panel 1: Wealth trajectory ---
        days = 1:length(result.wealth);
        p1 = plot(days, result.wealth ./ B₀, label="EWLS", linewidth=2, color=:steelblue,
            ylabel="W/W₀", title="Path $(p): Wealth Trajectory")
        hline!(p1, [1.0], linestyle=:dot, color=:gray50, label="")

        # --- Panel 2: Alpha drift for first two tickers ---
        t1, t2 = my_tickers[1], my_tickers[2];
        α_1 = [h[1] for h in result.param_history[t1]];
        α_2 = [h[1] for h in result.param_history[t2]];
        p2 = plot(0:length(α_1)-1, α_1, label="α $(t1)", linewidth=1.5, color=:coral,
            ylabel="α (annualized)", title="Path $(p): Alpha Drift", xlabel="Trading Day")
        plot!(p2, 0:length(α_2)-1, α_2, label="α $(t2)", linewidth=1.5, color=:goldenrod)

        display(plot(p1, p2, layout=(2,1), size=(800, 450)))
        println("Path $(p): W/W₀ = $(round(result.wealth[end]/B₀, digits=3))")
    end
end;

___
## Task 3: Half-Life Sensitivity
In this task, we sweep three half-life values ($h \in \{21, 63, 126\}$ days) on a single path to see the bias-variance trade-off in action. Short half-lives track regime shifts fast but amplify noise; long half-lives are stable but slow. Each sweep value calls [`replay_engine_ewls(...)`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.replay_engine_ewls) with that half-life.

> __What should we see?__
>
> The $h = 21$ estimates should be noisy but responsive. The $h = 126$ estimates should be smooth but lag behind regime transitions. The $h = 63$ default should sit between the two.

In the code block below, we run the sweep on path 1 and overlay the three wealth trajectories and α paths on a two-panel figure.

In [ ]:
let
    # --- Step 1: Pick a single path ---
    p_idx = 1;
    mkt = scenario.market_paths[p_idx, :];
    T_total = scenario.n_steps;
    pmatrix = zeros(T_total, N + 1);
    pmatrix[:, 1] = 1:T_total;
    for k in 1:N
        pmatrix[:, k + 1] = scenario.price_paths[p_idx, :, k];
    end

    rules_params = (max_drawdown = TRIGGER_MAX_DRAWDOWN, max_turnover = TRIGGER_MAX_TURNOVER);
    half_lives = [21.0, 63.0, 126.0];
    colors = [:coral, :steelblue, :green];
    labels = ["h=21", "h=63", "h=126"];

    # --- Panel 1: Wealth across half-lives ---
    p1 = plot(title="Half-Life Sweep: Wealth (Path $(p_idx))", ylabel="W/W₀", size=(800, 300));
    # --- Panel 2: Alpha drift across half-lives ---
    p2 = plot(title="Half-Life Sweep: α Drift ($(my_tickers[1]))", ylabel="α", xlabel="Trading Day", size=(800, 250));

    # --- Step 2: Run EWLS at each half-life ---
    for (j, h) in enumerate(half_lives)
        result = replay_engine_ewls(pmatrix, mkt, my_tickers, sim_params, rules_params;
            B₀ = B₀, offset = offset, half_life = h, prior_weight = h,
            N_short = L_short, N_long = L_long, GAIN = GAIN, N_growth = L_growth);

        plot!(p1, 1:length(result.wealth), result.wealth ./ B₀,
            label=labels[j], linewidth=2, color=colors[j])

        α_hist = [h_val[1] for h_val in result.param_history[my_tickers[1]]];
        plot!(p2, 0:length(α_hist)-1, α_hist,
            label=labels[j], linewidth=1.5, color=colors[j])

        println("h=$(Int(h)):  W/W₀ = $(round(result.wealth[end]/B₀, digits=3))")
    end

    hline!(p1, [1.0], linestyle=:dot, color=:gray50, label="")
    display(plot(p1, p2, layout=grid(2,1, heights=[0.55, 0.45]), size=(800, 500)))
end;

___
## Summary
This example replayed the Cobb-Douglas engine with EWLS parameter updates on a 500-path Monte Carlo ensemble, zoomed into individual paths to visualize parameter drift, and swept the half-life to illustrate the bias-variance trade-off. The ensemble results are saved to `ewls-replay-results.jld2` for the validation report in Example 3.

> __Key Takeaways:__
>
> * __Online updates improve distributional performance:__ The EWLS engine adapts its SIM parameters to regime shifts, producing a tighter wealth distribution and fewer failure paths than the frozen baseline. The improvement concentrates on paths that cross regime boundaries during the trading horizon.
> * __Single paths reveal the adaptation mechanism:__ Alpha and beta estimates drift visibly at regime boundaries, showing the engine's world-model updating in real time. Wealth diverges from the frozen baseline precisely where the parameter trajectories pull apart.
> * __Half-life controls the speed-stability trade-off:__ Short half-lives track regime shifts quickly but amplify noise, while long half-lives are stable but slow to adapt. The default half-life used in this example sits between the two and is the value the engine ships with.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.